# Generate answers for behavior evaluation
GPU required. Generate with fixed decoding from the final checkpoints of the selected no-replay, random, and MFR runs. The output JSONL is later used by automatic evaluators and blinded review. Preference-pair accuracy alone does not measure open-ended behavior.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
import pandas as pd
if not os.path.exists('/content/mfr-dpo'):
    !git clone -q https://github.com/prabudhd2003/mfr-dpo.git /content/mfr-dpo
elif subprocess.run(['git', '-C', '/content/mfr-dpo', 'status', '--porcelain', '--', 'data/v2'], capture_output=True, text=True, check=True).stdout.strip():
    print('Keeping local data/v2; skipping git pull so it is not overwritten.')
else:
    !git -C /content/mfr-dpo pull --ff-only -q
from google.colab import drive
drive.mount('/content/drive')
!pip install -q -r /content/mfr-dpo/requirements-eval.txt
REPO = '/content/mfr-dpo'
DRIVE_DIR = '/content/drive/MyDrive/CSCI544/mfr-dpo'
sys.path.insert(0, f'{REPO}/src')
import mfr_data, mfr_dpo, mfr_eval
from mfr_utils import load_protocol
protocol = load_protocol(f'{REPO}/configs/experiment_protocol.json')
splits = mfr_data.load_splits(f'{REPO}/data/v2')
RUN_NAMES = ['v2_o2_none_s0', 'v2_o2_random_s0', 'v2_o2_mfr_s0']  # replace with selected matched runs


In [ ]:
for run_index, run_name in enumerate(RUN_NAMES, start=1):
    run_dir = Path(DRIVE_DIR) / 'runs' / run_name
    mfr_eval.assert_test_ready(run_dir)
    settings = json.load(open(run_dir / 'settings.json'))
    adapter = run_dir / f"stage3_{settings['order'][-1]}"
    print(f'[{run_index}/{len(RUN_NAMES)}] Loading {run_name}...', flush=True)
    model, tokenizer = mfr_dpo.load_model(protocol['model_name'], protocol['lora_r'], adapter_path=adapter, revision=protocol['model_revision'], lora_alpha=protocol['lora_alpha'], lora_dropout=protocol['lora_dropout'])
    prompts = pd.concat([parts['test'][['id','prompt']].assign(behavior=name) for name, parts in splits.items()], ignore_index=True)
    generations = mfr_eval.generate_responses(model, tokenizer, prompts, **protocol['generation'], seed=settings['seed'])
    generations['method'] = settings['method']
    generations = generations.merge(prompts[['id','behavior']], on='id')
    output = run_dir / 'generation' / 'test_generations.jsonl'
    mfr_eval.save_generations(generations, output, {'run_name': run_name, 'decoding': protocol['generation']})
    print('saved', output)
    del model


In [ ]:
# Fixed external instruction-following benchmark; results and samples are saved per run.
for run_name in RUN_NAMES:
    run_dir = str(Path(DRIVE_DIR) / 'runs' / run_name)
    command = [sys.executable, '-u', f'{REPO}/scripts/run_ifeval.py', '--run-dir', run_dir]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    while True:
        chunk = process.stdout.read(4096)
        if not chunk:
            break
        print(chunk.decode('utf-8', errors='replace'), end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise RuntimeError(f'IFEval stopped with exit code {return_code}; see the error above.')


IFEval is an external instruction-following check. Also apply a versioned safety evaluator to the saved safety generations and preserve its per-prompt outputs. Never use these final results to retune the method.